# **ML-проект «Титаник»**

### Библиотеки

In [22]:
import shutil
import subprocess
import sys
from pathlib import Path

# import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler 

### Константы / Конфиг

In [23]:
# -----------------------------
# Константы воспроизводимости
# -----------------------------
RANDOM_STATE = 42          # фиксируем случайность для повторяемых результатов
N_SPLITS = 5               # число фолдов в кросс-валидации

# -----------------------------
# Константы качества данных
# -----------------------------
MISSING_THRESHOLD = 30     # порог (%) для выделения признаков с большим числом пропусков

# -----------------------------
# Параметры модели
# -----------------------------
LOGREG_MAX_ITER = 2000     # максимум итераций оптимизатора LogisticRegression 

### Воспроизводимость

1. В корне проекта выполните: `pip install -r requirements.txt`
2. На [Kaggle → Settings → API](https://www.kaggle.com/settings) нажмите **Create New Token**, получите `kaggle.json` и положите его в `%USERPROFILE%\.kaggle\` (Windows) или `~/.kaggle/` (Linux/macOS).

После установки пакетов **перезапустите ядро**. Ячейка ниже при отсутствии файлов скачает данные соревнования в каталог `data/`.

In [24]:
_cwd = Path.cwd().resolve()

DATA_DIR = (
    _cwd.parent / "data" if _cwd.name == "notebooks" else _cwd / "data"
).resolve()

DATA_DIR.mkdir(
    parents=True, 
    exist_ok=True,
)

if not (
    DATA_DIR / "train.csv"
).is_file():
    scripts = Path(sys.executable).parent / "Scripts"
    kaggle_cmd = None
    for name in (
        "kaggle.exe", 
        "kaggle"
    ):
        p = scripts / name
        if p.is_file():
            kaggle_cmd = str(p)
            break
    if kaggle_cmd is None:
        kaggle_cmd = shutil.which("kaggle")
    if kaggle_cmd is None:
        raise RuntimeError(
            "Не найден kaggle CLI. Выполните: pip install -r requirements.txt "
            "(из корня проекта) и перезапустите ядро."
        )
    subprocess.run(
        [
            kaggle_cmd,
            "competitions",
            "download",
            "-c",
            "titanic",
            "-p",
            str(DATA_DIR),
            "--unzip",
        ],
        check=True,
    )
    print("Данные загружены в", DATA_DIR)
else:
    print("Данные уже есть:", DATA_DIR) 

Данные уже есть: D:\Рабочий стол\titanic_project\data


### Описание датасета

#### Обучающая выборка **`train.csv`**

Обучающую выборку нужно использовать для построения моделей машинного обучения. Для неё мы даём **исход** (также называемый **истиной**, *ground truth*) для каждого пассажира. Модель будет опираться на **признаки** (*features*), например пол и класс пассажира. Можно также применять **конструирование признаков** (*feature engineering*), чтобы создавать новые признаки.

#### Тестовая выборка **`test.csv`**

Тестовую выборку нужно использовать, чтобы оценить, насколько хорошо модель работает на **новых данных**, которые она не видела при обучении. Для тестовой выборки мы **не** даём истинные ответы по каждому пассажиру — их нужно **предсказать вам**. Для каждого пассажира в тестовой выборке используйте обученную модель, чтобы предсказать, **выжил** ли он при крушении «Титаника» или нет.

#### Пример файла отправки

Также в комплекте есть файл **`gender_submission.csv`** — набор предположений, что выживают **все женщины и только они**. Это пример того, **как должен выглядеть файл с отправкой** (*submission*).

### Описание признаков

| Признак | Расшифровка | Ключ |
|------------|-------------|------------------|
| **Survived** | Выживание | 0 = нет, 1 = да |
| **Pclass** | Класс билета | 1 = 1-й класс, 2 = 2-й класс, 3 = 3-й класс |
| **Sex** | Пол | |
| **Age** | Возраст в годах | |
| **SibSp** | Число братьев/сестёр / супругов на борту «Титаника» | |
| **Parch** | Число родителей / детей на борту «Титаника» | |
| **Ticket** | Номер билета | |
| **Fare** | Стоимость проезда (пассажирский тариф) | |
| **Cabin** | Номер каюты | |
| **Embarked** | Порт посадки | C = Шербур, Q = Куинстаун, S = Саутгемптон |

#### Примечания к признакам

- **`Pclass`** — по смыслу близок к социально-экономическому статусу (SES): 1-й класс — «верхний», 2-й — «средний», 3-й — «нижний».
- **`Age`** — возраст может быть дробным, если меньше одного года. Оценочный возраст часто записан в виде `xx.5`.
- **`SibSp`** — в данных: *sibling* — брат, сестра, сводные брат/сестра; *spouse* — муж или жена (любовницы и женихи/невесты не учитывались).
- **`Parch`** — *parent* — мать, отец; *child* — дочь, сын, падчерица/пасынок. Дети, ехавшие только с няней, могут иметь **Parch = 0**.

### Импорт данных

In [25]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

In [26]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [27]:
test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


### Общие статистики

In [28]:
print("train shape:", train.shape)
print("test shape :", test.shape)

train shape: (891, 12)
test shape : (418, 11)


In [29]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


### Пропуски

In [30]:
# Доля пропусков по всем колонкам
missing_pct = train.isnull().mean() * 100

# Только проблемные колонки
problem_cols = missing_pct[missing_pct >= MISSING_THRESHOLD].sort_values(ascending=False)

print(f"Колонки с пропусками >= {MISSING_THRESHOLD}%:")
for col, pct in problem_cols.items():
    print(f"- {col}: {pct:.2f}%") 

Колонки с пропусками >= 30%:
- Cabin: 77.10%


### Вывод по пропускам

При пороге `MISSING_THRESHOLD = 30%` проблемной является только колонка `Cabin` (~77.1% пропусков).

In [31]:
train.describe() 

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Сохраняем ID теста — он нужен для submission

In [32]:
test_ids = test["PassengerId"]

Целевая переменная отдельно

In [33]:
y = train["Survived"] 

### Формируем `X_train` и `X_test`

На первом этапе строим **базовую модель** на простых и устойчивых признаках:
`Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, `Embarked`.

Почему сейчас **не включаем** `Ticket`:

- **`Ticket`** — неструктурированный текст с большим числом уникальных значений; без feature engineering может добавить шум.

- Цель текущего шага — получить **чистый baseline**, от которого удобно измерять улучшения.

Почему **не включаем** `Cabin`:

- **`Cabin`** содержит очень много пропусков, он не пригоден для модели.

In [34]:
feature_cols = [
    "Pclass", "Sex", "Age", "SibSp", 
    "Parch", "Fare", "Embarked"
]

X_train = train[feature_cols]
X_test = test[feature_cols] 

Разделяем числовые и категориальные признаки

In [35]:
num_features = ["Age", "SibSp", "Parch", "Fare"]
cat_features = ["Pclass", "Sex", "Embarked"] 

### Pipeline Preprocessing

Преобразование числовых признаков

In [36]:
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]) 

Преобразование категориальных признаков

In [37]:
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
]) 

Маршрутизация по колонкам

In [38]:
preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_features),
    ("cat", cat_transformer, cat_features),
]) 

Финальный Pipeline: preprocessing + model

In [39]:
pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=LOGREG_MAX_ITER)),
]) 

### Кросс-валидация

In [40]:
cv = StratifiedKFold(
    n_splits=N_SPLITS, 
    shuffle=True, 
    random_state=RANDOM_STATE
)

scores = cross_val_score(
    pipe, 
    X_train, 
    y, 
    cv=cv, 
    scoring="accuracy"
)

cv_mean = scores.mean()
cv_std = scores.std()

print(f"CV accuracy: {cv_mean:.4f} +/- {cv_std:.4f}")

CV accuracy: 0.7969 +/- 0.0146


### Базовая модель

In [41]:
pipe.fit(X_train, y)
pred_baseline = pipe.predict(X_test) 

### Submission базовой модели

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": pred_baseline
})

In [ ]:
assert submission.shape[0] == test.shape[0], "Ошибка: число строк в submission не равно числу строк в test."
assert list(submission.columns) == ["PassengerId", "Survived"], "Ошибка: неверные колонки или их порядок."
assert set(submission["Survived"].unique()).issubset({0, 1}), "Ошибка: в Survived есть значения вне {0, 1}."

In [ ]:
submission.head() 

In [ ]:
submission.to_csv(
    DATA_DIR / "submission_baseline.csv", index=False
) 

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


### Вывод по базовой модели

- Модель: `LogisticRegression` + `Pipeline` (`ColumnTransformer`).

- CV-оценка: `accuracy = 0.7969 +/- 0.0146` (5-fold).

- Public score на Kaggle: `0.76555`.

- Базовый pipeline валиден и используется как точка отсчета для следующих итераций инженерии признаков.